In [ ]:
### Script to integrate, preprocess and harmonize all available data sets
### In our use-case
# Single Cell RNA Seq
# Cytokine Data
# Neutrophil Data
# Clinical Data
# Proteomics

#############################################
# Prerequisites - Load Libraries

In [1]:
source('MS1_Functions.r')

In [2]:
### Inform about execution start
popup_function_pos('02_Integrate_and_Normalize_Data_Sources: Execution Started')

In [3]:
source('MS0_Libraries.r')

[1] "/ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env/lib/R/library/"


Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”
Warning message:
“package ‘tibble’ was built under R version 4.3.3”
Warning message:
“package ‘purrr’ was built under R version 4.3.3”
Warning message:
“package ‘stringr’ was built under R version 4.3.3”
Warning message:
“package ‘forcats’ was built under R version 4.3.3”
Warning message:
“package ‘lubridate’ was built under R version 4.3.3”
── Attaching core tidyverse packages ──────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.3     ✔ tibble    3.2.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
“package ‘backports’ w

In [4]:
source('MS2_Plot_Config.r')

Warning message:
“The `size` argument of `element_line()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.”


###############################################
# Preqrequisites Configurations & Parameters

In [5]:
### Load the parameters that are set via the configuration files

In [6]:
### Load configurations file
global_configs = read.csv('configurations/Data_Configs.csv', sep = ',')

In [7]:
head(global_configs,3)

,parameter,value
,<chr>,<chr>
1,data_path,/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/input_data/
2,result_path,/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results/
3,configuration_name,CGS_v5


In [8]:
data_path = global_configs$value[global_configs$parameter == 'data_path']

In [9]:
data_path

[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/input_data/"

In [10]:
result_path = global_configs$value[global_configs$parameter == 'result_path']

In [11]:
result_path

[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results/"

In [12]:
## Load the configuration file specifying single-cell specific filtering options

In [15]:
sc_configs = read.csv('configurations/02_Pre_Processing_Configs_SC.csv', sep = ',')

Warning message in read.table(file = file, header = header, sep = sep, quote = quote, :
“incomplete final line found by readTableHeader on 'configurations/02_Pre_Processing_Configs_SC.csv'”


In [16]:
head(sc_configs,2)

,configuration_name,data_name,data_type,cell_expr_thres1,cell_expr_thres2,cell_type_exclusion
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,CGS_v3,Prepared_sc_Data,.h5ad,15;5,10;10,"Erythrocytes,Plasma cells,Metabolically active Monocytes,Platelets"


In [17]:
sc_configs = sc_configs[sc_configs$data_name != '',]

In [18]:
sc_configs

,configuration_name,data_name,data_type,cell_expr_thres1,cell_expr_thres2,cell_type_exclusion
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,CGS_v3,Prepared_sc_Data,.h5ad,15;5,10;10,"Erythrocytes,Plasma cells,Metabolically active Monocytes,Platelets"


In [19]:
## Load the configuration file specifying the pre-processing options for all datasets

In [20]:
data_configs = read.csv('configurations/02_Pre_Processing_Configs.csv', sep = ',')

Warning message in read.table(file = file, header = header, sep = sep, quote = quote, :
“incomplete final line found by readTableHeader on 'configurations/02_Pre_Processing_Configs.csv'”


In [21]:
data_configs = data_configs[data_configs$configuration_name != '',]   # remove lines with empty configuration names
data_configs = data_configs[!is.na(data_configs$configuration_name),]  # remove lines with NA in configuration name

In [22]:
head(data_configs)

,configuration_name,data_name,file_type,data_type,remove_sample_ids,sample_filtering_thres,feature_filtering_thres,library_adjustment,log_transformation,variable_genes_filtering,quantile_normalization_samples,ribosomal_mitochondrial_gene_filtering,feature_wise_quantile_normalization
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<lgl>,<lgl>,<int>,<lgl>,<lgl>,<lgl>
1,CGS_v3,Prepared_Clinical_Data,.csv,clinical,HF4,1,0.20,FALSE,TRUE,1,FALSE,FALSE,TRUE
2,CGS_v3,Prepared_Proteomic_Data,.csv,proteomic,HF4,1,0.20,FALSE,FALSE,1,TRUE,FALSE,TRUE
3,CGS_v3,Prepared_sc_Data,.h5ad,single_cell,HF4,1,0.05,TRUE,TRUE,1,TRUE,TRUE,TRUE


In [23]:
### Generate the result data directory if it does not exist yet
if(!file.exists(paste0(result_path, '02_results'))){
    dir.create(file.path(paste0(result_path, '02_results')))
    }

# Load Data

In [24]:
### Load sc Data and exclude cluster_ids as specified in the configuration file

In [25]:
datasets = list()

In [26]:
## Load sc data (pseudobulk) generated in previous step
if(nrow(sc_configs) > 0){
for(j in 1:nrow(sc_configs)){
    sc_data_name = sc_configs$data_name[j]
    file_path = paste0(result_path, '/01_results/01_', sc_data_name, 'Pseudobulk_Table', '.csv')
    
    if(file.exists(file_path)){
        sc_data =  fread(file_path)

        sc_data$V1 = NULL

        ## Split up sc to different configs
        for(i in unique(sc_configs$configuration_name)){    
            for(j in unique(sc_configs$data_name[sc_configs$configuration_name == i])){

                data = sc_data[sc_data$dataset == j,]

                ## Exclude cluster_id's (cell-type clusters)
                if(!is.na(sc_configs$cell_type_exclusion[sc_configs$configuration_name == i])){
                data = data[!data$type %in% unlist(strsplit(sc_configs$cell_type_exclusion[sc_configs$configuration_name == i] ,',')),]
                    }

                datasets[[i]][[j]] = data
                }
            }
        popup_function_pos('Single Cell data Loaded')
        }
    else{
        print('No single-cell data loaded: check whether 01_Prepare_Pseudobulk.ipynb has been executed successfully ')
        popup_function_neg('No single-cell data loaded: check whether 01_Prepare_Pseudobulk.ipynb has been executed successfully ')
      }  
        
    }    
 }   

In [27]:
sc_data_name

[1] "Prepared_sc_Data"

In [28]:
#str(datasets)

In [29]:
length(unique(data$sample_id))

[1] 96

In [30]:
### Load the other datasets specified in the configuration file

In [31]:
for(i in unique(data_configs$configuration_name)){     # for each config
    for(j in unique(data_configs$data_name[data_configs$configuration_name == i])){      # each specifiec data-name
        
        configuration = data_configs[(data_configs$configuration_name == i) & (data_configs$data_name == j),]
        
        if(configuration$file_type == 'csv' | configuration$file_type == '.csv'){
            file_path = paste0(data_path, j, '.csv')
            if(file.exists(file_path)){
            
                data = read.csv(paste0(data_path, j, '.csv'))
                data$X = NULL
                data = melt(data, id.vars = 'sample_id')
                data$dataset = j
                data$type = configuration$data_type

                datasets[[i]][[j]] = data
                popup_function_pos(paste0('Data ', j, ' has been loaded successfully'))
                
                }
               else{ popup_function_neg(paste0('Error: Data ', j, ' could not be loaded. Check the data name and file path'))}
               
            
        }
        }
    }

In [32]:
#head(data,2)

In [33]:
#str(datasets)

In [34]:
data_backup = datasets # in case something should be re-executed, so loading of data is not necessary a second time

In [35]:
datasets = data_backup

In [36]:
#str(datasets)

# Pre-Process each dataset as specified in the configuration files

## Sample Filter

In [37]:
### Filter out sample_id's specified in the configuration file

In [38]:
head(data_configs,2)

,configuration_name,data_name,file_type,data_type,remove_sample_ids,sample_filtering_thres,feature_filtering_thres,library_adjustment,log_transformation,variable_genes_filtering,quantile_normalization_samples,ribosomal_mitochondrial_gene_filtering,feature_wise_quantile_normalization
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<lgl>,<lgl>,<int>,<lgl>,<lgl>,<lgl>
1,CGS_v3,Prepared_Clinical_Data,.csv,clinical,HF4,1,0.2,FALSE,TRUE,1,FALSE,FALSE,TRUE
2,CGS_v3,Prepared_Proteomic_Data,.csv,proteomic,HF4,1,0.2,FALSE,FALSE,1,TRUE,FALSE,TRUE


In [39]:
for(i in 1:nrow(data_configs)){
    ### Remove samples based on specified samples in remove_sample_ids column
    if( (!is.na(data_configs$remove_sample_ids[i])) & (data_configs$remove_sample_ids[i] != '')){
        
        print(paste0('Filtered specific samples for ',data_configs$data_name[i], ' ',  unique( unlist(strsplit(data_configs$remove_sample_ids[i], ',')))))
        
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        
        ### remove samples
        data = data[! data$sample_id %in% unlist(strsplit(data_configs$remove_sample_ids[i], ',')),]  # TBD check!
        
        ### replace adjusted data
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data
        
        }
    
     ### Remove samples based on threshold in sample_filtering_thres
     if ( (as.numeric(data_configs$sample_filtering_thres[i]) < 1) & (as.numeric(data_configs$sample_filtering_thres[i]) > 0)){
         
         print(paste0('Filtered samples based on threshold for ',data_configs$data_name[i])) 
         data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
         print(paste0('Amount samples before filtering ', length(unique(data$sample_id))))
         
         ### calculate percentage of features with zero values
         data = data %>% group_by(sample_id, type) %>% mutate(zero_expression_percentage = sum(value == 0)/ n())
         ### filter out samples if percentage higher than threshold
         data = data[data$zero_expression_percentage < data_configs$sample_filtering_thres[i],]
         print(paste0('Amount samples after filtering ', length(unique(data$sample_id))))
         popup_function_pos(paste0('Filtered samples for: ', data_configs$data_name[i], ' Amount samples after filtering ', length(unique(data$sample_id))))
         
         
         ### remove generated columns
         data$zero_expression_percentage = NULL
         
         ### replace adjusted data
         datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data            
    } 
}
        

[1] "Filtered specific samples for Prepared_Clinical_Data HF4"
[1] "Filtered specific samples for Prepared_Proteomic_Data HF4"
[1] "Filtered specific samples for Prepared_sc_Data HF4"


In [40]:
#str(datasets)

## Feature Removal (based on sample expression)

In [41]:
## Filter out features that are not expressed in a certain amount of sample (threshold set in the configuration file)

In [42]:
head(data_configs,2)

,configuration_name,data_name,file_type,data_type,remove_sample_ids,sample_filtering_thres,feature_filtering_thres,library_adjustment,log_transformation,variable_genes_filtering,quantile_normalization_samples,ribosomal_mitochondrial_gene_filtering,feature_wise_quantile_normalization
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<lgl>,<lgl>,<int>,<lgl>,<lgl>,<lgl>
1,CGS_v3,Prepared_Clinical_Data,.csv,clinical,HF4,1,0.2,FALSE,TRUE,1,FALSE,FALSE,TRUE
2,CGS_v3,Prepared_Proteomic_Data,.csv,proteomic,HF4,1,0.2,FALSE,FALSE,1,TRUE,FALSE,TRUE


In [43]:
data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]

In [44]:
head(data,2)

sample_id,variable,value,dataset,type
<chr>,<chr>,<dbl>,<chr>,<chr>
HF10,A1BG,0.01179825,Prepared_sc_Data,Activated T cells
HF11,A1BG,0.02166085,Prepared_sc_Data,Activated T cells


In [45]:
for(i in 1:nrow(data_configs)){

    if( (!is.na(data_configs$feature_filtering_thres[i])) & (data_configs$feature_filtering_thres[i] != '')  & (data_configs$feature_filtering_thres[i] > 0)){
        
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        
        print(paste0(data_configs$configuration_name[i], ' ' ,data_configs$data_name[i]))
        
        ## Determine data to filter
        data$expression = TRUE
        data$expression[data$value == 0] = FALSE
        expression_filter = data %>% group_by(type, variable) %>% summarise(perc_expression = sum(expression)  )
        expression_filter$perc_expression = expression_filter$perc_expression / length(unique(data$sample_id))
        
        ## Apply filter
        filtered_out = expression_filter[expression_filter$perc_expression <= data_configs$feature_filtering_thres[i],]
        print(paste0( 'Filtered: ' ))
        if(nrow(filtered_out) > 0){
            print((head(filtered_out %>% dplyr::group_by(type) %>% dplyr::count())))
            }
        expression_filter = expression_filter[expression_filter$perc_expression >data_configs$feature_filtering_thres[i],]  # kept data
        
        data = merge(data, expression_filter[,c('type', 'variable')], by.x = c('type', 'variable'), by.y = c('type', 'variable'))   # filter the data
        
        ## Remove expression column 
        data$expression = NULL
        
        ## Replace 
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]  = data
        
      }
}

[1] "CGS_v3 Prepared_Clinical_Data"


`summarise()` has grouped output by 'type'. You can override using the `.groups` argument.


[1] "Filtered: "
[1] "CGS_v3 Prepared_Proteomic_Data"


`summarise()` has grouped output by 'type'. You can override using the `.groups` argument.


[1] "Filtered: "
[1] "CGS_v3 Prepared_sc_Data"


`summarise()` has grouped output by 'type'. You can override using the `.groups` argument.


[1] "Filtered: "
# A tibble: 6 × 2
# Groups:   type [6]
  type                        n
  <chr>                   <int>
1 Activated T cells        9255
2 B Cells                  9155
3 Classical Monocytes      7660
4 Dendritic cells          7100
5 NK cells                 6590
6 Non-Classical Monocytes 14834


In [46]:
names(datasets)

[1] "CGS_v3"

## Library Adjustment

In [47]:
## Normalize measured counts for each sample to have the same amount of counts

In [48]:
head(data_configs,2)

,configuration_name,data_name,file_type,data_type,remove_sample_ids,sample_filtering_thres,feature_filtering_thres,library_adjustment,log_transformation,variable_genes_filtering,quantile_normalization_samples,ribosomal_mitochondrial_gene_filtering,feature_wise_quantile_normalization
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<lgl>,<lgl>,<int>,<lgl>,<lgl>,<lgl>
1,CGS_v3,Prepared_Clinical_Data,.csv,clinical,HF4,1,0.2,FALSE,TRUE,1,FALSE,FALSE,TRUE
2,CGS_v3,Prepared_Proteomic_Data,.csv,proteomic,HF4,1,0.2,FALSE,FALSE,1,TRUE,FALSE,TRUE


In [49]:
for(i in 1:nrow(data_configs)){
    if((data_configs$library_adjustment[i] == 'TRUE')){
        
        print(paste0('Library Adjustment for ',data_configs$data_name[i]))
        popup_function_pos(paste0('Library Adjustment for ',data_configs$data_name[i]))
        
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]

        ### Calculate scaling factor per sample
        data = data %>% group_by(sample_id,type) %>% mutate(sample_counts = sum(value))
        data = data %>% group_by(type) %>% mutate(mean_sample_counts = mean(sample_counts))
        
        data$scaling_factor = data$sample_counts/ data$mean_sample_counts
        data$scaling_factor[data$scaling_factor == 0] = 1 # avoid dividing by 0; TBD whether to include or exclude samples with only zero counts in a cell-type
        
        ### Apply scaling to counts
        
        data$value = data$value / data$scaling_factor
        
        ### Save transformed data to list
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data
        
        }
    }
        

[1] "Library Adjustment for Prepared_sc_Data"


In [50]:
names(datasets)

[1] "CGS_v3"

## Gene Filtering (according to cells expressing genes - only for sc Data)

In [51]:
### Remove genes from the single-cell dataset that are expressed in a too low amount of cells

In [52]:
head(sc_configs,2)

,configuration_name,data_name,data_type,cell_expr_thres1,cell_expr_thres2,cell_type_exclusion
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,CGS_v3,Prepared_sc_Data,.h5ad,15;5,10;10,"Erythrocytes,Plasma cells,Metabolically active Monocytes,Platelets"


In [53]:
## Load gene filtering information from previous script

In [54]:
gene_expression_info = data.frame()

In [55]:
for(i in sc_configs$data_name){
    data= read.csv(paste0(result_path, '/01_results/01_' ,i, '_Gene_Expr_per_Cell_Type.csv'))
    data$X = NULL
    
    data$data_name = i
    gene_expression_info = rbind(gene_expression_info, data)
    }

In [56]:
head(gene_expression_info,2)

,perc_cells_expressing_gene,total_amount_cells_expressing_gene,gene,cluster,dataset,data_name
,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>
1,0.01555694,4,AL627309.1,T cells,Prepared_sc_Data,Prepared_sc_Data
2,0.01944617,5,AL627309.5,T cells,Prepared_sc_Data,Prepared_sc_Data


In [57]:
sc_configs

,configuration_name,data_name,data_type,cell_expr_thres1,cell_expr_thres2,cell_type_exclusion
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,CGS_v3,Prepared_sc_Data,.h5ad,15;5,10;10,"Erythrocytes,Plasma cells,Metabolically active Monocytes,Platelets"


In [58]:
if(nrow(sc_configs) > 0){
for(i in unique(gene_expression_info$data_name)){
        
        print(paste0('Gene Filtering for  ',sc_configs$configuration_name[sc_configs$data_name == i]))
        popup_function_pos(paste0('Gene Filtering for  ', i))
        
        data = datasets[[sc_configs$configuration_name[sc_configs$data_name == i]]][[i]]
        gene_expr_data = gene_expression_info[gene_expression_info$data_name == i,]
    
        ## Get thresholds for config
        thres1 = as.numeric(unlist(str_split(sc_configs$cell_expr_thres1[sc_configs$data_name == i], ';')))
        thres2 = as.numeric(unlist(str_split(sc_configs$cell_expr_thres2[sc_configs$data_name == i], ';')))
        amount_samples = length(unique(data$sample_id))
        print(paste0('Amount Samples', amount_samples))
    
        ## Filter down gene based on the expression info
        gene_filtering =  gene_expr_data[((     gene_expr_data$perc_cells > thres1[1]) & (     gene_expr_data$total_amount_cells_expressing_gene > amount_samples * thres1[2])) |
         ((     gene_expr_data$perc_cells > thres2[1]) & (     gene_expr_data$total_amount_cells_expressing_gene > amount_samples * thres2[2])) ,]
    
        ## Apply to thresholds set in the configuration file
        filtered_data = data.frame()
        for( k in unique(data$type)){
            data_cluster = data[(data$type == k) & (data$variable %in% gene_filtering$gene[gene_filtering$cluster == k]),]
            filtered_data = rbind(filtered_data, data_cluster)
            }
        datasets[[sc_configs$configuration_name[sc_configs$data_name == i]]][[i]] = filtered_data
    }
    }

[1] "Gene Filtering for  CGS_v3"
[1] "Amount Samples95"


In [59]:
### Amount of genes after filtering

In [60]:
#unique(datasets[[1]][['Prepared_sc_Data']][,c('type', 'variable')])%>% group_by(type) %>% dplyr::count()

In [61]:
head(filtered_data,2)

type,variable,sample_id,value,dataset,sample_counts,mean_sample_counts,scaling_factor
<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>
Activated T cells,AAK1,HF10,0.1100287,Prepared_sc_Data,520.6127,511.0685,1.0186748
Activated T cells,AAK1,HF11,0.0819107,Prepared_sc_Data,473.0228,511.0685,0.9255564


In [62]:
#str(datasets)

## Log Transformation

In [63]:
## Apply log transformation to the data types specified in the configuration file

In [64]:
head(data_configs,2)

,configuration_name,data_name,file_type,data_type,remove_sample_ids,sample_filtering_thres,feature_filtering_thres,library_adjustment,log_transformation,variable_genes_filtering,quantile_normalization_samples,ribosomal_mitochondrial_gene_filtering,feature_wise_quantile_normalization
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<lgl>,<lgl>,<int>,<lgl>,<lgl>,<lgl>
1,CGS_v3,Prepared_Clinical_Data,.csv,clinical,HF4,1,0.2,FALSE,TRUE,1,FALSE,FALSE,TRUE
2,CGS_v3,Prepared_Proteomic_Data,.csv,proteomic,HF4,1,0.2,FALSE,FALSE,1,TRUE,FALSE,TRUE


In [65]:
data_configs

,configuration_name,data_name,file_type,data_type,remove_sample_ids,sample_filtering_thres,feature_filtering_thres,library_adjustment,log_transformation,variable_genes_filtering,quantile_normalization_samples,ribosomal_mitochondrial_gene_filtering,feature_wise_quantile_normalization
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<lgl>,<lgl>,<int>,<lgl>,<lgl>,<lgl>
1,CGS_v3,Prepared_Clinical_Data,.csv,clinical,HF4,1,0.20,FALSE,TRUE,1,FALSE,FALSE,TRUE
2,CGS_v3,Prepared_Proteomic_Data,.csv,proteomic,HF4,1,0.20,FALSE,FALSE,1,TRUE,FALSE,TRUE
3,CGS_v3,Prepared_sc_Data,.h5ad,single_cell,HF4,1,0.05,TRUE,TRUE,1,TRUE,TRUE,TRUE


In [66]:
for(i in 1:nrow(data_configs)){
    if((data_configs$log_transformation[i] == 'TRUE')){
        
        print(paste0('Log Transformation for ',data_configs$data_name[i]))
        popup_function_pos(paste0('Log Transformation for ',data_configs$data_name[i]))
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        
        data$value = log2(data$value + 1)  # add pseudocount of 1
        
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data # save adjusted data
        }
    }
        
        

[1] "Log Transformation for Prepared_Clinical_Data"
[1] "Log Transformation for Prepared_sc_Data"


## Variable Gene Filtering

In [67]:
### Filter on highly variable genes if specified in the configuration file

In [68]:
head(data_configs,2)

,configuration_name,data_name,file_type,data_type,remove_sample_ids,sample_filtering_thres,feature_filtering_thres,library_adjustment,log_transformation,variable_genes_filtering,quantile_normalization_samples,ribosomal_mitochondrial_gene_filtering,feature_wise_quantile_normalization
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<lgl>,<lgl>,<int>,<lgl>,<lgl>,<lgl>
1,CGS_v3,Prepared_Clinical_Data,.csv,clinical,HF4,1,0.2,FALSE,TRUE,1,FALSE,FALSE,TRUE
2,CGS_v3,Prepared_Proteomic_Data,.csv,proteomic,HF4,1,0.2,FALSE,FALSE,1,TRUE,FALSE,TRUE


In [69]:
for(i in 1:nrow(data_configs)){
    ### Filter genes with lowest variance
    if ( (as.numeric(data_configs$variable_genes_filtering[i]) < 1) & (as.numeric(data_configs$variable_genes_filtering[i]) > 0)){
        print(paste0('Variable Genes Filtering for ',data_configs$data_name[i]))
        popup_function_pos(paste0('Variable Genes Filtering for ',data_configs$data_name[i]))
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        
        ### Calculate variance and threshold
        data = data %>% group_by(variable, type) %>% mutate(feature_variance = var(value)) # variance
        data = data %>% group_by(type) %>% mutate(variance_threshold = quantile(feature_variance, probs = seq(0, 1, 0.01), na.rm = FALSE,
         names = TRUE)[(1-as.numeric(data_configs$variable_genes_filtering[i]))*100])   # threshold
        
        ### Filter
        data = data[data$feature_variance > data$variance_threshold,]
        
        ### remove generated columns
        data$feature_variance = NULL
        data$variance_threshold = NULL
        
        ### Save transformed data to list
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data
        
        }
    }
        

## Sample Quantile Normalization

In [70]:
### Apply Sample Quantile Normalization for the data-types specified in the configuration file

In [71]:
head(data_configs,2)

,configuration_name,data_name,file_type,data_type,remove_sample_ids,sample_filtering_thres,feature_filtering_thres,library_adjustment,log_transformation,variable_genes_filtering,quantile_normalization_samples,ribosomal_mitochondrial_gene_filtering,feature_wise_quantile_normalization
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<lgl>,<lgl>,<int>,<lgl>,<lgl>,<lgl>
1,CGS_v3,Prepared_Clinical_Data,.csv,clinical,HF4,1,0.2,FALSE,TRUE,1,FALSE,FALSE,TRUE
2,CGS_v3,Prepared_Proteomic_Data,.csv,proteomic,HF4,1,0.2,FALSE,FALSE,1,TRUE,FALSE,TRUE


In [72]:
for(i in 1:nrow(data_configs)){
    if((data_configs$quantile_normalization_samples[i] == 'TRUE')){
        
        print(paste0('Sample Quantile Normalization for ',data_configs$data_name[i]))
        popup_function_pos(paste0('Sample Quantile Normalization for ',data_configs$data_name[i]))
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        transformed_data = data.frame()
        
        for(k in unique(data$type)){
            data_type = data[data$type == k,]
            data_type = data_type %>% dcast(variable ~ sample_id, value.var = 'value')
            features = data_type$variable
            rownames(data_type) = features
            data_type$variable = NULL
            data_type = data_type[,colSums(is.na(data_type)) != nrow(data_type)] # remove na samples
            data_type  = quantile_normalization(data_type ) 
            data_type = data.frame(data_type)
            data_type$variable = features
            data_type = melt(data_type)
            colnames(data_type) = c('variable', 'sample_id', 'value')
            
            data_type$type = k 
            data_type$dataset = data_configs$data_name[i]
            transformed_data = rbind(transformed_data, data_type)
            
            }
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = transformed_data
        }
    }
            
        
        
        

[1] "Sample Quantile Normalization for Prepared_Proteomic_Data"


Using variable as id variables



[1] "Sample Quantile Normalization for Prepared_sc_Data"


Using variable as id variables

Using variable as id variables

Using variable as id variables

Using variable as id variables

Using variable as id variables

Using variable as id variables

Using variable as id variables

Using variable as id variables



## Gene Removal (ribosomal, mitochondrial)

In [73]:
### Remove ribosomal and mitochondrial genes (only works if 'Gene' annotation is given as SYMBOL

In [74]:
head(data_configs,2)

,configuration_name,data_name,file_type,data_type,remove_sample_ids,sample_filtering_thres,feature_filtering_thres,library_adjustment,log_transformation,variable_genes_filtering,quantile_normalization_samples,ribosomal_mitochondrial_gene_filtering,feature_wise_quantile_normalization
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<lgl>,<lgl>,<int>,<lgl>,<lgl>,<lgl>
1,CGS_v3,Prepared_Clinical_Data,.csv,clinical,HF4,1,0.2,FALSE,TRUE,1,FALSE,FALSE,TRUE
2,CGS_v3,Prepared_Proteomic_Data,.csv,proteomic,HF4,1,0.2,FALSE,FALSE,1,TRUE,FALSE,TRUE


In [75]:
for(i in 1:nrow(data_configs)){
    if((data_configs$ribosomal_mitochondrial_gene_filtering[i] == 'TRUE')){
        
        print(paste0('Remove ribosomal and mitochondrial genes for ',data_configs$data_name[i]))
        popup_function_pos(paste0('Remove ribosomal and mitochondrial genes for ',data_configs$data_name[i]))
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        
        ## Remove ribosomal and mitochondiral genes
        data = data[is.na(str_extract(data$variable, '^MT.*|^RPL.*|^RPS.*')),]
        
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data
        }
    }
        

        
        

[1] "Remove ribosomal and mitochondrial genes for Prepared_sc_Data"


# Merge data types and process

In [76]:
### Combine all the datasets to one dataset

In [77]:
#str(datasets)

In [78]:
### Combine all types into one dataset
datasets = lapply(datasets, function(x){
    data = do.call(rbind, x)
    })

In [79]:
### Overview amount of features per type/ view 

In [80]:
unique(datasets[[1]][,c('type', 'variable')]) %>% group_by(type) %>% dplyr::count()

type,n
<chr>,<int>
Activated T cells,886
B Cells,1178
Classical Monocytes,1538
Dendritic cells,1273
NK cells,1604
Non-Classical Monocytes,667
Regulatory T cells,1149
T cells,755
clinical,13


# Feature Wise Quantile Normalization

In [81]:
### Apply feature wise quantile normalization if specified in the configuration file

In [82]:
head(data_configs,2)

,configuration_name,data_name,file_type,data_type,remove_sample_ids,sample_filtering_thres,feature_filtering_thres,library_adjustment,log_transformation,variable_genes_filtering,quantile_normalization_samples,ribosomal_mitochondrial_gene_filtering,feature_wise_quantile_normalization
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<lgl>,<lgl>,<int>,<lgl>,<lgl>,<lgl>
1,CGS_v3,Prepared_Clinical_Data,.csv,clinical,HF4,1,0.2,FALSE,TRUE,1,FALSE,FALSE,TRUE
2,CGS_v3,Prepared_Proteomic_Data,.csv,proteomic,HF4,1,0.2,FALSE,FALSE,1,TRUE,FALSE,TRUE


In [83]:
data_configs$feature_wise_quantile_normalization

[1] TRUE TRUE TRUE

In [84]:
names(datasets)

[1] "CGS_v3"

In [85]:
for(i in names(datasets)){
    
    data = datasets[[i]]
    data$ident = paste0(data$type, '_0_', data$variable)
    final_data = dcast(data, sample_id ~ ident , value.var = "value") # ! with this merging there might be NA values for some samples on some data types
    rownames(final_data) = final_data$sample_id
    final_data$sample_id = NULL
    
    # Remove samples with only NA's
    data_nas = is.na(final_data)
    rowSums(data_nas)  # TBD maybe plot amount of NA per sample
    keep_samples = names(rowSums(data_nas))[rowSums(data_nas) != ncol(final_data)]
    final_data = final_data[keep_samples,]
    data_nas = data_nas[keep_samples,]
    
    # Feature Wise Quantile Normalization in kind of TRUE value
    if(unique(data_configs$feature_wise_quantile_normalization[data_configs$configuration_name == i]) == 'TRUE'){
        print('Applying Feature Wise Quantile Normalization')
        popup_function_pos(paste0('Applying Feature Wise Quantile Normalization',i))
        final_data = apply(final_data, 2,stdnorm)
        final_data = data.frame(final_data)
        final_data[data_nas] = NA
        final_data$sample_id = rownames(final_data)
        data_long = melt(final_data)
        data_long$type = str_extract(data_long$variable, '.*_0_')
        data_long$type  = str_replace(data_long$type , '_0_', '')
        data_long$variable = str_replace(data_long$variable, '.*_0_', '')
        datasets[[i]] = data_long
        
        }
    
    }
        
        
        

[1] "Applying Feature Wise Quantile Normalization"


Using sample_id as id variables



In [86]:
head(datasets[[1]],2)

,sample_id,variable,value,type
,<chr>,<chr>,<dbl>,<chr>
1,HF10,AAK1,0.4022501,Activated.T.cells
2,HF11,AAK1,0.0000000,Activated.T.cells


# Save the data

In [87]:
### Save the data to use as input in the next script

In [88]:
### Adjust variable names
datasets = lapply(datasets, function(x){
    x$gene = x$variable
    x$variable = paste0(x$type, '__', x$variable)
    return(x)
    })

In [89]:
for(i in names(datasets)){
    write.csv(datasets[[i]], paste0(result_path, '/02_results/02_Combined_Data_', i, '_INTEGRATED',  '.csv'))
    popup_function_pos(paste0('Saved', result_path, '/02_results/' , ' 02_Combined_Data_', i, '_INTEGRATED',  '.csv'))
    }

In [90]:
### Example of structure of dataset

In [91]:
head(datasets[[1]],2)

,sample_id,variable,value,type,gene
,<chr>,<chr>,<dbl>,<chr>,<chr>
1,HF10,Activated.T.cells__AAK1,0.4022501,Activated.T.cells,AAK1
2,HF11,Activated.T.cells__AAK1,0.0000000,Activated.T.cells,AAK1


In [92]:
length(unique(datasets[[1]]$sample_id))

[1] 99

# Update Configuration File

In [93]:
paste0(unique(datasets[[i]]$type), collapse = ',')  # TBD make config specific

[1] "Activated.T.cells,B.Cells,Classical.Monocytes,clinical,Dendritic.cells,NK.cells,Non.Classical.Monocytes,proteomic,Regulatory.T.cells,T.cells"

In [94]:
### Adjust 06 Pathway Analysis Configs

In [95]:
configs06 = data.frame(
    mofa_result_name = paste0(unique(data_configs$configuration_name), '_MOFA'),
    factor_set = '1',
    coverage_par = 0.2,
    types = paste0(unique(datasets[[i]]$type), collapse = ','),
    coverage_plot  = 0.5,
    p_value_plot = 0.05,
    max_pathways_plot = 8,
    enrichment_plot = 'negative',
    top_features_plot = 0.125,
    pathway_selection = '')

In [96]:
configs06

mofa_result_name,factor_set,coverage_par,types,coverage_plot,p_value_plot,max_pathways_plot,enrichment_plot,top_features_plot,pathway_selection
<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<chr>
CGS_v3_MOFA,1,0.2,"Activated.T.cells,B.Cells,Classical.Monocytes,clinical,Dendritic.cells,NK.cells,Non.Classical.Monocytes,proteomic,Regulatory.T.cells,T.cells",0.5,0.05,8,negative,0.125,


In [97]:
write.csv(configs06, 'configurations/06_Pathway_Configs.csv', row.names = FALSE)

In [98]:
### Inform about successful execution
popup_function_pos('02_Integrate_and_Normalize_Data_Sources: Execution Finished')

In [99]:
Sys.sleep(20)
popup_function_info('02_Integrate_and_Normalize_Data_Sources')